# MLP Gating Analysis

Analyze MLP gating: activation sparsity, pre-post correlation,
distribution statistics, and gating selectivity.

## Why This Matters

MLP gating provides tools for understanding how feed-forward layers transform and store information. Since MLPs have been shown to function as key-value memories, understanding their internal mechanics is essential for locating knowledge and understanding factual recall.

**Key references:**
- [Geva et al. (2021) "Transformer Feed-Forward Layers Are Key-Value Memories"](https://arxiv.org/abs/2012.14913) — Shows MLPs function as key-value memory stores
- [Bills et al. (2023) "Language Models Can Explain Neurons"](https://openaipublic.blob.core.windows.net/neuron-explainer/paper/index.html) — Automated neuron interpretation methods

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.mlp_gating_analysis import (
    mlp_activation_sparsity, mlp_pre_post_correlation,
    mlp_activation_distribution, mlp_gating_selectivity,
    mlp_gating_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Activation Sparsity

In [ ]:
for layer in range(4):
    result = mlp_activation_sparsity(model, tokens, layer=layer)
    bar = '█' * int(result['sparsity'] * 40)
    print(f"  Layer {layer}: sparsity={result['sparsity']:.1%}, "
          f"mean_mag={result['mean_magnitude']:.4f} {bar}")

## Pre-Post Correlation

In [ ]:
for layer in range(4):
    result = mlp_pre_post_correlation(model, tokens, layer=layer)
    print(f"  Layer {layer}: mean_corr={result['mean_correlation']:.4f}")

## Activation Distribution

In [ ]:
for layer in range(4):
    result = mlp_activation_distribution(model, tokens, layer=layer)
    print(f"  Layer {layer}: mean={result['mean']:.4f}, std={result['std']:.4f}, "
          f"skew={result['skewness']:.4f}, kurt={result['kurtosis']:.4f}")

## Gating Selectivity

In [ ]:
for layer in range(4):
    result = mlp_gating_selectivity(model, tokens, layer=layer, top_k=3)
    most = [(n, f'{s:.3f}') for n, s in result['most_selective']]
    print(f"  Layer {layer}: mean_sel={result['mean_selectivity']:.4f}, top={most}")

## Summary

In [ ]:
result = mlp_gating_summary(model, tokens)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: sparse={p['sparsity']:.1%}, "
          f"skew={p['skewness']:.4f}")